In [ ]:
import h5py

In [ ]:
PERTURBED_SWEEP_H5 = "/nfs-share/pa511/code_bases/new_jac/onnx_models/simple_stories/perturbed_sweep_small.h5"

with h5py.File(PERTURBED_SWEEP_H5, "r") as h5f:
    print(list(h5f.keys()))


In [ ]:
import numpy as np
import functools

# Row order is (sample_idx, pos, vocab_id) unravelled in that order (see
# scripts/generate_perturbed_sweep.py:build_chunk_rows), so reshaping
# straight back to (n_samples, seq, vocab_size, d_model) restores the
# original sweep structure -- no need to sort by the index arrays.
@functools.lru_cache(maxsize=None)
def load_vec_grid(layer: str) -> np.ndarray:
    with h5py.File(PERTURBED_SWEEP_H5, "r") as h5f:
        grp = h5f[layer]
        vec = grp["vec"][:]
        sample_idx = grp["sample_idx"][:]
        pos = grp["pos"][:]
        vocab_id = grp["vocab_id"][:]

    n_samples = sample_idx.max() + 1
    seq = pos.max() + 1
    vocab_size = vocab_id.max() + 1
    d_model = vec.shape[-1]

    grid = vec.reshape(n_samples, seq, vocab_size, d_model)
    assert grid.shape[0] * grid.shape[1] * grid.shape[2] == vec.shape[0]
    # sanity check: row 0 of the flat array is (sample_idx=0, pos=0, vocab_id=0)
    assert np.array_equal(grid[0, 0, 0], vec[0])
    return grid


# load_vec_grid("add_36").shape


In [ ]:
# load_vec_grid(layer)[sample_idx, pos] is the full vocab-sweep block
# (vocab_size, d_model) of `layer`'s outputs for that sequence/position --
# i.e. one row per candidate token substituted at `pos` in sequence
# `sample_idx`. Pass a layer name (e.g. "add_36") to pull that layer's
# sweep; call it with two consecutive layer names to compare activations
# across layers for the same perturbation.
def full_vocab_sweep(layer: str, sample_idx: int, pos: int) -> np.ndarray:
    return load_vec_grid(layer)[:, pos].reshape(-1, 384)  # (vocab_size, d_model)

data_36 = full_vocab_sweep("add_36", 64, 7)
data_38 = full_vocab_sweep("add_38", 64, 7)
data_36.shape, data_38.shape


In [ ]:
# Pairwise distances between every pair of points *within* each cloud
# separately -- data_36[i] vs data_36[j] for all i, j (and same for
# data_38), not data_36 vs data_38.
from scipy.spatial.distance import pdist, squareform


# indeces = np.arange(data_36.shape[0])
samples = np.random.choice(data_36.shape[0], (10000), replace=False)
print(samples)
# pdist already returns the condensed (upper-triangular, no diagonal, no
# duplicates) form directly -- squareform(pdist(x)) just expands that back
# out to the full symmetric matrix, so the flat vector of unique pairwise
# distances is pdist's own output, no need to re-flatten the square matrix.
flat_dists_36 = pdist(data_36[samples])  # (vocab_size * (vocab_size - 1) / 2,)
flat_dists_38 = pdist(data_38[samples])  # (vocab_size * (vocab_size - 1) / 2,)


In [ ]:
import matplotlib.pyplot as plt

plt.scatter(flat_dists_36, flat_dists_38)

In [ ]:
# flat_dists_36 and flat_dists_38 come from pdist on the same point
# ordering (both indexed by vocab id), so pair i in one lines up with pair i
# in the other -- the ratio is elementwise, no re-indexing needed.
#
# ||f(x) - f(y)|| / ||x - y|| for every pair (x, y) is a local estimate of
# the lower Lipschitz bound of the add_36 -> add_38 map: the minimum of
# this ratio over all pairs is how much the map is guaranteed to expand
# distances by, at least (co-Lipschitz / lower Lipschitz constant).
lipschitz_lower_ratios = flat_dists_38 / flat_dists_36

lipschitz_lower_ratios.shape, lipschitz_lower_ratios.min(), lipschitz_lower_ratios.mean(), lipschitz_lower_ratios.max()


In [ ]:
from scipy.stats import norm, gamma, kstest

g_full = lipschitz_lower_ratios

# Fit Normal and Gamma (Poisson doesn't apply -- it's discrete/non-negative
# integers only, and g_full is a continuous ratio) to the full distribution.
# norm_mu, norm_sigma = norm.fit(g_full)
# gamma_a, gamma_loc, gamma_scale = gamma.fit(g_full)

xs = np.linspace(g_full.min(), g_full.max(), 500)

plt.hist(g_full, bins=60, density=True, color='steelblue', edgecolor='black', alpha=0.6, label='empirical')
# plt.plot(xs, norm.pdf(xs, norm_mu, norm_sigma), color='crimson', lw=2, label=f'Normal (mu={norm_mu:.3g}, sigma={norm_sigma:.3g})')
# plt.plot(xs, gamma.pdf(xs, gamma_a, gamma_loc, gamma_scale), color='darkorange', lw=2, label=f'Gamma (a={gamma_a:.3g})')

plt.grid(True, alpha=0.3)
plt.xlabel('lipschitz_lower_ratio')
plt.ylabel('density')
plt.title(f'lipschitz_lower_ratios distribution (N={g_full.size})')
plt.legend()
plt.show()


In [ ]:
# Goodness of fit: one-sample KS test compares the empirical CDF of
# g_full against each fitted distribution's CDF. Lower KS statistic (D) and
# higher p-value both indicate a better fit; a small p-value (e.g. < 0.05)
# means the data significantly deviates from that fitted distribution.
ks_norm = kstest(g_full, norm.cdf, args=(norm_mu, norm_sigma))
ks_gamma = kstest(g_full, gamma.cdf, args=(gamma_a, gamma_loc, gamma_scale))

print(f"Normal fit: D={ks_norm.statistic:.4g}, p={ks_norm.pvalue:.4g}")
print(f"Gamma  fit: D={ks_gamma.statistic:.4g}, p={ks_gamma.pvalue:.4g}")
print(f"-> {'Gamma' if ks_gamma.statistic < ks_norm.statistic else 'Normal'} fits better (lower KS statistic)")


def p_less_than(x: float) -> dict:
    """P(X < x) under each fitted distribution, plus the empirical fraction
    of observed data below x for comparison."""
    return {
        "empirical": float(np.mean(g_full < x)),
        "normal": float(norm.cdf(x, norm_mu, norm_sigma)),
        "gamma": float(gamma.cdf(x, gamma_a, gamma_loc, gamma_scale)),
    }


p_less_than(1.0)


In [ ]:
# From-scratch plot (not reusing scripts/estimate_colipschitz_gpd.py's
# plot_sigma_min) so every step here is easy to check by eye.
#
# Left subplot: full density of lipschitz_lower_ratios, threshold u marked.
# Right subplot: just the left tail (exceedances, flipped back to the
# original ratio scale) -- empirical density vs. the fitted GPD density,
# with the GPD's estimated finite lower endpoint c_hat marked as the point
# where the fitted curve's support ends (the "intersection" of the GPD
# curve with the x-axis).
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import genpareto, gaussian_kde

g = lipschitz_lower_ratios

# Threshold u: POT rule -- u is set so exactly the n_tail_min smallest
# values of g fall below it (same default as fit_gpd_tail's n_tail_min).

g_sorted = np.sort(g)


In [ ]:
# n_tail_min = 0.025
# print(g_sorted[0])
# print(g_sorted[-1])
# threshold = g_sorted[0]+(g_sorted[-1]-g_sorted[0])*n_tail_min

threshold = 1.155546530142284
print(threshold)
# u = g_sorted[g_sorted<threshold]


In [ ]:
plt.plot(g, 'ro', markersize=3, alpha=0.3)
plt.axhline(threshold, color='black', linestyle='--', label=f'threshold={threshold:.4g}')
plt.grid(True, alpha=0.3)
plt.xlabel('pair index')
plt.ylabel('lipschitz_lower_ratio')
plt.legend()
plt.show()


In [ ]:
plt.hist(g[g < threshold], bins=30, color='steelblue', edgecolor='black', alpha=0.8)
plt.grid(True, alpha=0.3)
plt.xlabel('lipschitz_lower_ratio')
plt.ylabel('count')
plt.title(f'Exceedances below threshold (n={np.sum(g < threshold)})')
plt.show()


In [ ]:
threshold = 1.0907435848317626
exceedances = threshold - g[g < threshold]
plt.hist(exceedances, bins=30, color='steelblue', edgecolor='black', alpha=0.8)
plt.grid(True, alpha=0.3)
plt.xlabel('lipschitz_lower_ratio')
plt.ylabel('count')
plt.title(f'Exceedances values (n={np.sum(g < threshold)})')
plt.show()


In [ ]:
xi, loc, sigma = genpareto.fit(exceedances, floc=0)
print(xi, loc, sigma)


In [ ]:
# When xi < 0, the fitted GPD has a finite upper endpoint for the
# exceedances at -sigma/xi (its support is [0, -sigma/xi]). Since
# exceedances = threshold - g, that caps the exceedance from above, which
# means g has a finite *lower* endpoint:
#   g_min_limit = threshold - (-sigma / xi) = threshold + sigma / xi
# i.e. the GPD tail model predicts g can never fall below this value --
# the estimated lower Lipschitz bound in the limit of infinite samples.
if xi < 0:
    c_hat = -sigma / xi  # finite upper endpoint of the exceedance distribution
    g_min_limit = threshold + sigma / xi
    print(f"xi={xi:.4g} < 0 -> finite endpoint: exceedance <= {c_hat:.4g}, g >= {g_min_limit:.4g}")
else:
    print(f"xi={xi:.4g} >= 0 -> no finite lower limit for g (unbounded tail)")


In [ ]:
# P(g > 0) under the fitted GPD tail model. g > 0 <=> exceedance = threshold
# - g < threshold, so this is the GPD CDF evaluated at z = threshold (the
# largest exceedance value consistent with g staying positive), combined
# with P(g >= threshold) = (1 - tail_fraction) from the empirical data
# above the threshold (the GPD only models the tail below threshold).
tail_fraction = exceedances.size / g.size  # P(g < threshold), empirical

# P(g <= 0) = P(exceedance >= threshold) = tail_fraction * P(Z >= threshold | GPD)
p_g_le_0 = tail_fraction * genpareto.sf(threshold, xi, loc=loc, scale=sigma)
p_g_gt_0 = 1 - p_g_le_0

print(f"P(g <= 0) = {p_g_le_0:.6g}")
print(f"P(g > 0)  = {p_g_gt_0:.6g}")


In [ ]:
# Overlay the fitted GPD pdf on the exceedance histogram -- density=True so
# the bars integrate to 1 and are directly comparable to the pdf curve.
# Restricted to exceedances > 0.08 (both the histogram data and the curve's
# x-range), not just an axis zoom.
z_min = 0.08
exceedances_tail = exceedances

plt.hist(exceedances_tail, bins=30, density=True, color='steelblue', edgecolor='black', alpha=0.6, label='exceedances (empirical)')

xs = np.linspace(0, exceedances.max(), 500)
plt.plot(xs, genpareto.pdf(xs, xi, loc=loc, scale=sigma), color='crimson', lw=2, label=f'fitted GPD (xi={xi:.3g}, sigma={sigma:.3g})')

plt.grid(True, alpha=0.3)
plt.xlabel('exceedance z = threshold - g')
plt.ylabel('density')
plt.title(f'Exceedances > {z_min} vs. fitted GPD')
plt.legend()
plt.show()


In [ ]:
# Sensitivity sweep: for each tail size n_tail (number of smallest g values
# used as exceedances), set threshold u = g_sorted[n_tail] and refit the GPD
# shape xi on those n_tail exceedances. This is the standard
# threshold-stability check for POT/GPD fits -- xi should stabilize (be
# roughly flat) over some range of n_tail if the fit is trustworthy.
#
# Also track the mean excess e(u) = mean(u - g[g < u]) at each threshold --
# the mean residual life plot. For a GPD tail, e(u) is linear in u, so a
# straight-line stretch here is further evidence the GPD is a good model
# over that range (and pairs with the xi-stability check above).
n_tail_values = np.arange(20, 20000, 100)
xi_sweep = np.full(n_tail_values.shape, np.nan)
mean_excess_sweep = np.full(n_tail_values.shape, np.nan)
threshold_sweep = np.full(n_tail_values.shape, np.nan)

for i, n_tail in enumerate(n_tail_values):
    u_n = g_sorted[n_tail]
    exc_n = u_n - g_sorted[:n_tail]  # the n_tail smallest g values, as exceedances
    xi_n, _loc_n, _sigma_n = genpareto.fit(exc_n, floc=0)
    xi_sweep[i] = xi_n
    mean_excess_sweep[i] = exc_n.mean()
    threshold_sweep[i] = u_n



In [ ]:

fig, ((ax_xi, ax_excess), (ax_xi_u, ax_excess_u)) = plt.subplots(2, 2, figsize=(13, 10))

ax_xi.plot(n_tail_values, xi_sweep, color='steelblue')
ax_xi.axhline(0, color='black', linestyle='--', alpha=0.6, label='xi = 0')
ax_xi.grid(True, alpha=0.3)
ax_xi.set_xlabel('n_tail (number of exceedances)')
ax_xi.set_ylabel('fitted xi')
ax_xi.set_title('GPD shape parameter xi vs. tail size')
ax_xi.legend()

# Mean residual life plot: threshold u on the x-axis (not n_tail), mean
# excess e(u) on the y-axis -- the conventional MRL orientation.
# ax_excess.plot(threshold_sweep, mean_excess_sweep, color='darkorange')
ax_excess.plot(n_tail_values, mean_excess_sweep, color='darkorange')

ax_excess.grid(True, alpha=0.3)
ax_excess.set_xlabel('threshold u')
ax_excess.set_ylabel('mean excess e(u)')
ax_excess.set_title('Mean residual life plot')

# Same two sweeps, but against threshold u instead of n_tail -- the
# conventional x-axis for both xi-stability and mean residual life plots.
ax_xi_u.plot(threshold_sweep, xi_sweep, color='steelblue')
ax_xi_u.axhline(0, color='black', linestyle='--', alpha=0.6, label='xi = 0')
ax_xi_u.grid(True, alpha=0.3)
ax_xi_u.set_xlabel('threshold u')
ax_xi_u.set_ylabel('fitted xi')
ax_xi_u.set_title('GPD shape parameter xi vs. threshold')
ax_xi_u.legend()

ax_excess_u.plot(threshold_sweep, mean_excess_sweep, color='darkorange')
ax_excess_u.grid(True, alpha=0.3)
ax_excess_u.set_xlabel('threshold u')
ax_excess_u.set_ylabel('mean excess e(u)')
ax_excess_u.set_title('Mean residual life plot vs. threshold')

fig.tight_layout()
plt.show()

In [ ]:
# Expected Quantile Discrepancy (Bader, Yan & Zhang 2018) for automatic
# threshold selection: for each candidate n_tail (equivalently u =
# g_sorted[n_tail]), fit a GPD to the n_tail exceedances, then measure the
# weighted L1 distance between the empirical exceedance quantiles and the
# fitted GPD's theoretical quantiles. Interior (near-median) quantile
# mismatches are weighted down and tail mismatches weighted up, since a
# good GPD fit should track the extreme tail well even if it's imperfect
# near the threshold itself. The n_tail minimizing this discrepancy is the
# EQD-selected threshold.
#
# Weight w_i = min(i/(k+1), 1 - i/(k+1)) is the standard EQD weight: it is
# 0 at the median rank and rises toward 0.5 at the two ends of the ordered
# sample, so extreme order statistics count most.
def eqd_statistic(exceedances: np.ndarray) -> float:
    k = exceedances.size
    exc_sorted = np.sort(exceedances)
    xi_k, _loc_k, sigma_k = genpareto.fit(exc_sorted, floc=0)

    ranks = np.arange(1, k + 1)
    p = ranks / (k + 1)
    theoretical_q = genpareto.ppf(p, xi_k, loc=0, scale=sigma_k)

    weights = np.minimum(p, 1 - p)
    return float(np.sum(weights * np.abs(exc_sorted - theoretical_q)) / k)


eqd_n_tail_values = np.arange(20, 20000, 100)
eqd_sweep = np.full(eqd_n_tail_values.shape, np.nan)

for i, n_tail in enumerate(eqd_n_tail_values):
    u_n = g_sorted[n_tail]
    exc_n = u_n - g_sorted[:n_tail]
    eqd_sweep[i] = eqd_statistic(exc_n)

eqd_best_idx = np.argmin(eqd_sweep)
eqd_best_n_tail = eqd_n_tail_values[eqd_best_idx]
eqd_best_threshold = g_sorted[eqd_best_n_tail]
print(f"EQD-selected n_tail={eqd_best_n_tail}, u={eqd_best_threshold:.4g}, EQD={eqd_sweep[eqd_best_idx]:.4g}")

fig, (ax_eqd_n, ax_eqd_u) = plt.subplots(1, 2, figsize=(13, 5))

ax_eqd_n.plot(eqd_n_tail_values, eqd_sweep, color='seagreen')
ax_eqd_n.axvline(eqd_best_n_tail, color='black', linestyle='--', alpha=0.6, label=f'min at n_tail={eqd_best_n_tail}')
ax_eqd_n.grid(True, alpha=0.3)
ax_eqd_n.set_xlabel('n_tail (number of exceedances)')
ax_eqd_n.set_ylabel('EQD statistic')
ax_eqd_n.set_title('Expected Quantile Discrepancy vs. tail size')
ax_eqd_n.legend()

ax_eqd_u.plot(g_sorted[eqd_n_tail_values], eqd_sweep, color='seagreen')
ax_eqd_u.axvline(eqd_best_threshold, color='black', linestyle='--', alpha=0.6, label=f'min at u={eqd_best_threshold:.4g}')
ax_eqd_u.grid(True, alpha=0.3)
ax_eqd_u.set_xlabel('threshold u')
ax_eqd_u.set_ylabel('EQD statistic')
ax_eqd_u.set_title('Expected Quantile Discrepancy vs. threshold')
ax_eqd_u.legend()

fig.tight_layout()
plt.show()
print(eqd_best_n_tail, eqd_best_threshold)

In [3]:
cnt = 4
for i in range(50):
    print('[[sublayers]]')
    if i % 2 ==0:
        print(f'input = "add_{cnt}"')
        cnt+=2
        print(f'output = "add_{cnt}"')

        
    else:
        print(f'input = "add_{cnt}"')
        cnt+=4
        print(f'output = "add_{cnt}"')
        
    print('loss_type = "MaskedSumLoss"')
    print('input_shape = [32, 8, 384]')
    print()
    
    

[[sublayers]]
input = "add_4"
output = "add_6"
loss_type = "MaskedSumLoss"
input_shape = [32, 8, 384]

[[sublayers]]
input = "add_6"
output = "add_10"
loss_type = "MaskedSumLoss"
input_shape = [32, 8, 384]

[[sublayers]]
input = "add_10"
output = "add_12"
loss_type = "MaskedSumLoss"
input_shape = [32, 8, 384]

[[sublayers]]
input = "add_12"
output = "add_16"
loss_type = "MaskedSumLoss"
input_shape = [32, 8, 384]

[[sublayers]]
input = "add_16"
output = "add_18"
loss_type = "MaskedSumLoss"
input_shape = [32, 8, 384]

[[sublayers]]
input = "add_18"
output = "add_22"
loss_type = "MaskedSumLoss"
input_shape = [32, 8, 384]

[[sublayers]]
input = "add_22"
output = "add_24"
loss_type = "MaskedSumLoss"
input_shape = [32, 8, 384]

[[sublayers]]
input = "add_24"
output = "add_28"
loss_type = "MaskedSumLoss"
input_shape = [32, 8, 384]

[[sublayers]]
input = "add_28"
output = "add_30"
loss_type = "MaskedSumLoss"
input_shape = [32, 8, 384]

[[sublayers]]
input = "add_30"
output = "add_34"
loss_type